# Notebook 5: Model Explainability & SHAP Analysis

Building an accurate churn prediction model is only half the battle. To drive **real business action**, we need to answer:

- **Which features** contribute most to churn predictions globally?
- **Why** did the model flag a specific customer as high-risk?
- **What should** a retention agent say to that customer?

This notebook uses **SHAP (SHapley Additive exPlanations)** to open the black box of our model and generate actionable, human-readable explanations.

### What is SHAP?

SHAP is based on **Shapley values** from cooperative game theory. For each prediction, SHAP computes the marginal contribution of each feature to the model's output:

- **Positive SHAP value** → feature pushes prediction toward churn (higher risk)
- **Negative SHAP value** → feature pushes prediction away from churn (lower risk)
- **Magnitude** → how strongly the feature influences this particular prediction

SHAP has several key properties:
- **Consistency**: If a feature's contribution increases, its SHAP value never decreases
- **Local accuracy**: SHAP values sum exactly to the model's output
- **Missingness**: Features not present contribute 0

For Logistic Regression, we use `shap.LinearExplainer` which computes exact SHAP values analytically — no sampling approximation needed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import shap
import joblib
import json
import sys
import os
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.abspath('../src'))

from data_preprocessing import prepare_features, add_engineered_features, clean_data, load_raw_data
from prediction import predict_single, build_full_recommendation_output

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
shap.initjs()

MODEL_DIR = '../models/'
REPORTS_DIR = '../reports/figures/'
RAW_DATA_PATH = '../data/raw/telco_churn.csv'
PROCESSED_DATA_PATH = '../data/processed/churn_processed.csv'

# Load trained model, preprocessing pipeline, and metadata
print('Loading model artifacts...')

model_path = os.path.join(MODEL_DIR, 'best_model.pkl')
pipeline_path = os.path.join(MODEL_DIR, 'preprocessing_pipeline.pkl')
metadata_path = os.path.join(MODEL_DIR, 'model_metadata.json')

model = joblib.load(model_path)
pipeline = joblib.load(pipeline_path)

with open(metadata_path, 'r') as f:
    metadata = json.load(f)

print(f'Model loaded: {metadata.get("model_name", "Unknown")}')
print(f'Trained on: {metadata.get("trained_at", "Unknown")}')
print(f'Test Recall: {metadata.get("test_recall", "Unknown")}')
print(f'Test ROC-AUC: {metadata.get("test_roc_auc", "Unknown")}')
print(f'Feature count: {metadata.get("n_features", "Unknown")}')

## Global Feature Importance

**Global feature importance** tells us which features have the **largest average impact** on predictions across all customers. This answers: "What does the model think matters most for predicting churn?"

For Logistic Regression, global importance can be derived from the absolute values of the model coefficients (after scaling). Features with larger absolute coefficients have stronger influence on the log-odds of churn.

We also plot the pre-computed `feature_importance.png` chart generated during model training, which shows a clean ranked bar chart.

In [ ]:
# Load and display the pre-generated feature importance chart
fi_path = os.path.join(REPORTS_DIR, 'feature_importance.png')

if os.path.exists(fi_path):
    img = mpimg.imread(fi_path)
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.imshow(img)
    ax.axis('off')
    plt.title('Global Feature Importance (Logistic Regression Coefficients)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    # Generate feature importance from model coefficients
    print(f'Feature importance chart not found at {fi_path}.')
    print('Generating from model coefficients...')

    feature_names = metadata.get('feature_names', [])
    if hasattr(model, 'coef_') and len(feature_names) > 0:
        coef = model.coef_[0]
        fi_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coef})
        fi_df['Abs Coefficient'] = fi_df['Coefficient'].abs()
        fi_df = fi_df.sort_values('Abs Coefficient', ascending=True).tail(20)

        fig, ax = plt.subplots(figsize=(10, 8))
        colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in fi_df['Coefficient']]
        ax.barh(fi_df['Feature'], fi_df['Coefficient'], color=colors, edgecolor='white')
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_title('Top 20 Feature Coefficients (Logistic Regression)', fontweight='bold')
        ax.set_xlabel('Coefficient Value (positive = increases churn risk)')
        plt.tight_layout()
        plt.savefig(fi_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Feature importance chart saved to {fi_path}')
    else:
        print('Could not extract feature importance — model may not have coef_ attribute.')

## SHAP Values

While feature importance shows **global patterns**, SHAP values tell us exactly **how much each feature contributed to each individual prediction**.

### SHAP Summary Plot

The SHAP summary plot (beeswarm plot) shows:
- **Y-axis**: Features sorted by total SHAP impact (most important at top)
- **X-axis**: SHAP value (positive = pushes toward churn, negative = pushes away)
- **Color**: Feature value (red = high feature value, blue = low feature value)
- **Each dot**: One customer

This allows us to see not just importance, but the **direction and distribution** of each feature's effect.

In [ ]:
# Load a sample of processed data and transform it
print('Loading and transforming data for SHAP analysis...')

df_processed = pd.read_csv(PROCESSED_DATA_PATH)

# Prepare features using the same pipeline
X, y = prepare_features(RAW_DATA_PATH)

# Use a sample of 500 for SHAP (LinearExplainer can handle full dataset but sample is faster)
np.random.seed(42)
sample_idx = np.random.choice(len(X), size=min(500, len(X)), replace=False)
X_sample = X.iloc[sample_idx]

# Transform the sample using the preprocessing pipeline (scaler only — model step excluded)
# If pipeline includes scaler + model, extract scaler step
if hasattr(pipeline, 'named_steps'):
    # Try to get scaler from pipeline steps
    scaler_steps = [name for name, step in pipeline.named_steps.items()
                    if hasattr(step, 'transform') and not hasattr(step, 'predict')]
    if scaler_steps:
        from sklearn.pipeline import Pipeline as SkPipeline
        scaler_pipeline = SkPipeline([(name, pipeline.named_steps[name])
                                       for name in scaler_steps])
        X_sample_transformed = scaler_pipeline.transform(X_sample)
    else:
        X_sample_transformed = X_sample.values
else:
    X_sample_transformed = pipeline.transform(X_sample)

print(f'Sample shape: {X_sample_transformed.shape}')

# Compute SHAP values using LinearExplainer
print('Computing SHAP values using LinearExplainer...')
explainer = shap.LinearExplainer(model, X_sample_transformed, feature_perturbation='correlation_dependent')
shap_values = explainer.shap_values(X_sample_transformed)

print(f'SHAP values computed. Shape: {shap_values.shape}')

# SHAP Summary Plot (beeswarm)
feature_names = X.columns.tolist()

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    X_sample_transformed,
    feature_names=feature_names,
    max_display=20,
    show=False
)
plt.title('SHAP Summary Plot — Top 20 Features by Impact', fontsize=13, fontweight='bold')
plt.tight_layout()
shap_summary_path = os.path.join(REPORTS_DIR, 'shap_summary.png')
plt.savefig(shap_summary_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'SHAP summary plot saved to {shap_summary_path}')

## Local Explanation — Single Customer

Local explanations answer the question: **"Why does the model think THIS specific customer will churn?"**

We demonstrate with a **high-risk customer profile**:
- Month-to-month contract (highest risk contract type)
- Tenure of only 3 months (very new customer)
- Fiber optic internet service (high-churn segment)
- Paying via electronic check (non-auto-pay)
- Monthly charges of $89 (above average)
- No add-on security or backup services

The **SHAP waterfall plot** shows exactly how each feature contribution builds up from the base rate to this customer's predicted churn probability.

In [ ]:
# Define a high-risk sample customer
sample_customer = {
    'gender': 'Male',
    'SeniorCitizen': 0,
    'Partner': 'No',
    'Dependents': 'No',
    'tenure': 3,
    'PhoneService': 'Yes',
    'MultipleLines': 'No',
    'InternetService': 'Fiber optic',
    'OnlineSecurity': 'No',
    'OnlineBackup': 'No',
    'DeviceProtection': 'No',
    'TechSupport': 'No',
    'StreamingTV': 'No',
    'StreamingMovies': 'No',
    'Contract': 'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 89.10,
    'TotalCharges': '267.30'
}

print('=== High-Risk Customer Profile ===')
for k, v in sample_customer.items():
    print(f'  {k}: {v}')

# Convert to DataFrame and add engineered features
customer_df = pd.DataFrame([sample_customer])
customer_df['TotalCharges'] = pd.to_numeric(customer_df['TotalCharges'], errors='coerce')
customer_df = add_engineered_features(customer_df)

# Align columns with training feature matrix
# One-hot encode the customer using the same structure
customer_encoded = pd.get_dummies(customer_df)

# Align to feature matrix columns
reference_cols = X.columns.tolist()
for col in reference_cols:
    if col not in customer_encoded.columns:
        customer_encoded[col] = 0
customer_encoded = customer_encoded[reference_cols]

# Transform using pipeline
if hasattr(pipeline, 'named_steps'):
    scaler_steps = [name for name, step in pipeline.named_steps.items()
                    if hasattr(step, 'transform') and not hasattr(step, 'predict')]
    if scaler_steps:
        from sklearn.pipeline import Pipeline as SkPipeline
        sp = SkPipeline([(n, pipeline.named_steps[n]) for n in scaler_steps])
        customer_transformed = sp.transform(customer_encoded)
    else:
        customer_transformed = customer_encoded.values
else:
    customer_transformed = pipeline.transform(customer_encoded)

# Get SHAP values for this customer
customer_shap = explainer.shap_values(customer_transformed)
base_value = explainer.expected_value

# Waterfall plot
print('\nGenerating SHAP waterfall plot for high-risk customer...')
shap_explanation = shap.Explanation(
    values=customer_shap[0],
    base_values=base_value,
    data=customer_transformed[0],
    feature_names=reference_cols
)

plt.figure(figsize=(10, 7))
shap.waterfall_plot(shap_explanation, max_display=15, show=False)
plt.title('SHAP Waterfall Plot — High-Risk Customer\n(tenure=3, month-to-month, fiber optic)',
          fontsize=12, fontweight='bold')
plt.tight_layout()
waterfall_path = os.path.join(REPORTS_DIR, 'shap_waterfall_sample.png')
plt.savefig(waterfall_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Waterfall plot saved to {waterfall_path}')

# Compute churn probability for this customer
churn_prob = model.predict_proba(customer_transformed)[0][1]
print(f'\nPredicted churn probability: {churn_prob:.1%}')
print(f'Risk level: {"HIGH" if churn_prob > 0.6 else "MEDIUM" if churn_prob > 0.35 else "LOW"}')

## Business Interpretation of Top Features

The SHAP analysis consistently highlights the following features as the most important churn drivers. Here is how to interpret each one and what business actions they suggest:

| Feature | Business Meaning | Recommended Action |
|---------|-----------------|-------------------|
| **Tenure** (low values → high churn risk) | New customers have not yet experienced enough value to develop loyalty. The "honeymoon period" is the most critical window. | Implement a structured **90-day onboarding program**: proactive check-ins at day 7, 30, and 90. Assign a dedicated customer success rep for the first 3 months. |
| **Contract type** (month-to-month → high risk) | Month-to-month customers have zero switching cost. They can cancel at any time with no financial penalty. | Offer a **"loyalty upgrade" discount** (e.g., 15% off for switching to annual contract). Highlight long-term savings during the first billing cycle. |
| **Monthly charges** (high charges → higher risk) | Customers paying premium rates who don't perceive equivalent value are price-sensitive churners. | Conduct a **value audit** for high-paying customers: do they use all subscribed services? Offer a right-sizing consultation to reduce bill while retaining the customer. |
| **Payment method** (electronic check → higher risk) | Manual payment customers are less "sticky" — no auto-renewal inertia. Also correlates with month-to-month contracts. | Offer an **auto-pay discount** (e.g., $5/month off) to migrate electronic check customers to bank transfer or credit card. |

### Reading the SHAP Waterfall Plot

For the high-risk customer shown above:
- **Starting point (base value)**: The average model output across all training customers (~26% churn probability, the base rate)
- **Each bar**: How much that feature shifts the prediction up (red, toward churn) or down (blue, away from churn)
- **Final value**: The customer's individual churn probability — significantly higher than the base rate due to stacked risk factors

## Recommendation Engine Demo

The final component of our system is a **recommendation engine** that generates personalized, actionable retention advice for each flagged customer.

Given a customer's profile and the model's prediction, the recommendation engine:
1. Identifies the **top risk factors** for this customer using SHAP values
2. Maps each risk factor to a **specific retention action**
3. Ranks actions by **estimated impact**
4. Returns a structured JSON response suitable for use by a CRM system or retention agent

The demo below shows the full output for the high-risk customer from the previous cell.

In [ ]:
# Run prediction and generate full recommendation output
print('=== Recommendation Engine Demo ===')
print('Customer: High-risk profile (tenure=3, month-to-month, fiber optic)')
print('-' * 60)

# Predict using the single-customer prediction function
prediction_result = predict_single(
    customer_data=sample_customer,
    model=model,
    pipeline=pipeline,
    feature_names=reference_cols
)

# Build full recommendation output
full_output = build_full_recommendation_output(
    customer_data=sample_customer,
    prediction_result=prediction_result,
    shap_values=customer_shap[0],
    feature_names=reference_cols
)

# Pretty print the JSON output
print(json.dumps(full_output, indent=2))

In [ ]:
# Visualize the recommendation output in a readable format
print('\n=== Retention Action Plan for Customer ===')
print(f"Customer ID: {full_output.get('customer_id', 'N/A')}")
print(f"Churn Probability: {full_output.get('churn_probability', 0):.1%}")
print(f"Risk Level: {full_output.get('risk_level', 'Unknown')}")
print()

recommendations = full_output.get('recommendations', [])
if recommendations:
    print('Top Retention Recommendations:')
    print('-' * 50)
    for i, rec in enumerate(recommendations, 1):
        print(f"{i}. [{rec.get('priority', 'N/A').upper()}] {rec.get('action', 'N/A')}")
        print(f"   Driver: {rec.get('churn_driver', 'N/A')}")
        print(f"   Expected Impact: {rec.get('expected_impact', 'N/A')}")
        if 'script' in rec:
            print(f"   Agent Script: \"{rec['script']}\"")
        print()
else:
    print('No recommendations generated.')
    print('Full output:')
    print(json.dumps(full_output, indent=2))

print('=' * 60)
print('Notebook 5 complete. End of Customer Churn Prediction project.')